# AICE Professional 주피터라이트 실전 모의고사 12회 — 공개데이터(온라인 쇼핑 구매 의도 · 문장 속 개체 관계(KLUE-RE) · 고양이 품종)

## [ 필독 ] AICE Professional 모의고사 유의사항

- 공식 샘플문항(가공식품 카테고리 분류)과 **같은 시험지 형식**으로 만든 모의고사입니다. 3문항, 180분 기준입니다.
- 공식 시험과 같은 **Tabular / Text / Image 3유형**입니다. 2번 Text 문항은 공식 샘플(상품명 → 카테고리)과 같은 텍스트 분류입니다.
- 본인 스스로 모델을 만들고 **달성 목표 이상의 성능**이 나오도록 개발해야 합니다. 정답(테스트 라벨)은 제공되지 않습니다.
  대신 이 노트북 **맨 마지막 셀**(전체 자동 채점, 정답 키 인코딩)로 제출 파일을 스스로 채점할 수 있습니다(연습용).

## 사용할 수 있는 라이브러리 (주피터라이트 판)

이 모의고사는 **브라우저 주피터라이트(Pyodide · Python 3.14)** 에서 설치 없이 끝까지 풀 수 있도록 라이브러리를 제한했습니다.
필요한 라이브러리는 주피터라이트에 **미리 설치**되어 있습니다. 맨 위 '브라우저 실습' 셀은 확인만 하고 바로 끝납니다(Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다).

| 구분 | 쓸 수 있음 | 쓰지 않음(브라우저에서 안 됨) |
|---|---|---|
| 데이터·그림 | numpy · pandas · matplotlib · seaborn · Pillow | — |
| 머신러닝 | scikit-learn · LightGBM · XGBoost · imbalanced-learn · joblib | CatBoost |
| 텍스트 | scikit-learn 의 CountVectorizer · TfidfVectorizer · 내장 영어 불용어, **NLTK**(불용어·토큰화·어간 추출, 데이터는 `/drive/nltk_data` 에 미리 들어 있음) | 형태소 분석기 · BERT 등 사전학습 언어 모델 |
| 딥러닝 | `tensorflow.keras` 문법(TensorFlow.js 로 실행되는 대체 구현체, 미리 설치): `Sequential`·함수형(`Input` → 층 → `Model`) · Dense · Dropout · BatchNormalization · Conv2D · MaxPooling2D · GlobalAveragePooling2D · Embedding · LSTM · GRU · 콜백 3종(EarlyStopping · ModelCheckpoint · ReduceLROnPlateau) · `load_img` · `img_to_array` · `image_dataset_from_directory` · `ImageDataGenerator` · `model.save` / `load_model` · **MobileNet v1 전이학습**(`alpha=0.25`, 224×224, `weights='imagenet'` 만) | PyTorch · 다른 사전학습 모델(MobileNetV2/V3 · ResNet · EfficientNet 등) · 사용자 정의 층·손실 · GPU |

- 이미지는 64×64 정도로 줄여 작은 CNN 을 직접 학습하거나, MobileNet v1(alpha 0.25)로 특징을 뽑아 분류기를 학습하세요(전이학습). 브라우저 TensorFlow.js 는 CPython TensorFlow 와 수치가 조금 달라 딥러닝 점수가 약간 다르게 나올 수 있습니다.
- 학습이 오래 걸리면 epoch 를 줄이거나 조기 종료(EarlyStopping)를 쓰세요. 최신 Chrome·Edge 를 권장합니다.

## 시간 배분(180분)

| 문항 | 권장 시간 | 먼저 할 일 |
|---|---|---|
| 1. Tabular (30점) | 50분 | `info()`·`describe()` 로 자료형·결측 확인 → 기본 모델로 제출 파일부터 만들기 |
| 2. Text (35점) | 60분 | 라벨 분포·문장 길이 확인 → TF-IDF + 선형 모델로 기준 점수 |
| 3. Image (35점) | 60분 | 압축 풀기·폴더 구조 확인 → 작은 CNN 으로 기준 점수 |
| 마무리 | 10분 | 제출 파일 6개 이름·열 구조 확인 → 맨 마지막 채점 셀 실행 |

## 시험 자원 가이드

| 구분 | 공식 시험 | 이 모의고사 |
|---|---|---|
| IDE | CPU 8 core · Memory 32G · GPU 1개(VRAM 16GB) | **브라우저 주피터라이트**(설치 없음, CPU) — Colab·로컬 Jupyter 에서도 그대로 실행됨 |
| SW | Python 3.10 · TensorFlow 2.13 · PyTorch 2.1 | Python 3.14(Pyodide) · scikit-learn · LightGBM · XGBoost · Keras 문법(TensorFlow.js) — 위 표의 라이브러리만 |

## 유의 사항

- 막히면 문항 아래 접혀 있는 **힌트 1·2·3단계**를 순서대로 펼쳐 보세요. 단계가 올라갈수록 정답에 가까워집니다.
- 제출 파일명은 **고정**입니다: `answer_1~3` 모델 파일과 `answer_1~3.csv`.


문제에 제시되는 가이드는 필수적으로 준수하여 주시기 바랍니다. 이를 준수하지 않을 경우 채점 제한 또는 0점 처리될 수 있으니
각별히 유의해 주시기 바랍니다.

#### **[학습 정보]**
- 학습 목표 : Tabular · Text · Image 예측 모델 3종을 만들어 달성 목표 이상의 예측 파일을 제출하고, 제출한 코드로 같은 모델이 재현되게 할 수 있다
- AICE 역량 : PT1·PT2 표 데이터 / PX1·PX2 텍스트 / PI1·PI2 이미지 / PS1 제출 파일 형식·재현성
- 제출 파일 : `answer_1~3` 모델 파일(.pkl · .h5 등)과 `answer_1~3.csv` — **파일명은 고정**입니다(이름을 바꾸면 채점되지 않음)

| 문항 | 문제 | 달성 목표 | 역량 | 기술 | 예상 시간 |
|---|---|---|---|---|---|
| 1. Tabular | 온라인 쇼핑몰 방문 세션의 구매 여부 예측 | F1-score 0.62 이상 | PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | pandas, scikit-learn(StratifiedKFold), LightGBM, joblib | 50분 |
| 2. Text | 문장 속 두 개체의 관계 분류(6종) | 정확도 0.9 이상 | PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | pandas, scikit-learn(ColumnTransformer, Pipeline, OneHotEncoder, TfidfVectorizer, LogisticRegression, LinearSVC), joblib | 60분 |
| 3. Image | 고양이 사진으로 품종 분류(5종) | 정확도 0.8 이상 | PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | pandas, scikit-learn(LogisticRegression), Keras(TensorFlow.js), joblib | 60분 |

In [1]:
# (브라우저 실습) 주피터라이트에는 tensorflow(TensorFlow.js 대체 구현체)·scikit-learn·LightGBM 등이 미리 설치되어 있어 이 셀은 확인만 합니다.
# Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다.
import os, sys, importlib.util
if 'pyodide' in sys.modules:
    os.environ['AICE_IMG_DIR'] = '/tmp/aice_images/' + os.path.basename(os.getcwd())   # 이미지 압축은 메모리 폴더에 풂(빠름)
    if not ('tensorflow' in sys.modules or importlib.util.find_spec('tensorflow')):
        # 미리 설치되지 않은 사이트(패치 전 주피터라이트)에서만: 같은 폴더의 tfjs_keras.py 를 불러옵니다
        sys.path[:0] = [os.getcwd(), '/drive']
        import tfjs_keras
        await tfjs_keras.load()
print('준비 완료 — 다음 셀로 넘어가세요.')

준비 완료 — 다음 셀로 넘어가세요.


## 1. Tabular 문제 - 온라인 쇼핑몰 방문 세션의 구매 여부 예측 (30점)

<sub>학습 목표: 온라인 쇼핑몰 방문 세션의 구매 여부 예측 모델로 테스트 F1-score 0.62 이상을 달성할 수 있다 | 역량: PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | 기술: pandas, scikit-learn(StratifiedKFold), LightGBM, joblib | 난이도: 실전 | 예상 시간: 50분</sub>

### 문제 개요

쇼핑몰 방문 세션의 페이지 열람 수·체류 시간·이탈률·특별일 근접도·방문자 유형 등으로 **그 세션에서 구매가 일어날지** 를 예측하는 이진 분류 문제입니다. 구매 세션이 약 15% 로 적어 **불균형 처리와 F1-score** 가 핵심입니다.

제공된 훈련 데이터셋을 활용해 이진 분류 모델을 개발하고, 테스트 데이터셋의 'Revenue' 값을 예측하시기 바랍니다.

### 달성 목표

"01_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **F1-score 0.62 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `01_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `01_test_x.csv`: 테스트 데이터셋입니다. 라벨('Revenue') 값이 제외되어 있습니다.

**Label 정보**

- 'Revenue' : 1 = 구매, 0 = 구매 안 함

**Data 정보**

- 'id' : 행 번호
- 'Administrative · Informational · ProductRelated (+ _Duration)' : 페이지 종류별 열람 수와 머문 시간(초)
- 'BounceRates · ExitRates · PageValues' : 이탈률 · 종료율 · 페이지 가치(구매로 이어진 페이지의 평균 가치)
- 'SpecialDay' : 특별일(어버이날·밸런타인데이 등)과 가까운 정도 0~1
- 'Month · VisitorType · Weekend' : 월(문자열) · 방문자 유형 · 주말 여부
- 'OperatingSystems · Browser · Region · TrafficType' : 숫자 코드로 된 범주

**데이터 출처** — Online Shoppers Purchasing Intention(UCI 468 · CC BY 4.0) — 1년간 방문 세션 12,330건, 구매 비율 약 15.5%

### 데이터 예시

`01_train.csv`

| id | Administrative | Administrative_Duration | Informational | Informational_Duration | ProductRelated | ProductRelated_Duration | BounceRates | … |
|---|---|---|---|---|---|---|---|---|
| 1 | 6 | 523.214 | 2 | 263 | 48 | 2309.73 | 0.00384615 | … |
| 2 | 2 | 20.2 | 0 | 0 | 14 | 1261.32 | 0 | … |
| 3 | 0 | 0 | 0 | 0 | 3 | 71.5 | 0.0666667 | … |

`01_test_x.csv`

| id | Administrative | Administrative_Duration | Informational | Informational_Duration | ProductRelated | ProductRelated_Duration | BounceRates | … |
|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 0 | 0 | 0 | 22 | 2189.5 | 0.00952381 | … |
| 2 | 0 | 0 | 0 | 0 | 49 | 2110.12 | 0 | … |
| 3 | 0 | 0 | 0 | 0 | 19 | 623.2 | 0.0222222 | … |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 온라인 쇼핑몰 방문 세션의 구매 여부 예측 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_1.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('Revenue') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_1.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'Revenue' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'Revenue' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- Month·VisitorType 은 문자열, Weekend 는 True/False 입니다. 원-핫 인코딩 또는 정수로 바꾸세요.
- 구매(1)가 약 15% 입니다. 평가 지표는 **F1-score(구매=1 기준)** 이므로 `class_weight='balanced'` 나 **분류 임계값 조정** 이 효과적입니다.
- 예측값은 0/1 정수로 제출하세요.

### 채점 진행 방식 안내

- 1번 문항 배점 : 30점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_1.csv 로 제출된 예측 데이터를 정답지와 대조하여, **F1-score 0.558 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [2]:

import pandas as pd

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')

print("훈련 데이터:", train.shape)
print("테스트 데이터:", test.shape)

print("\n[자료형 및 결측 확인]")
train.info()

print("\n[정답 분포]")
print(train['Revenue'].value_counts())
print(train['Revenue'].value_counts(normalize=True).round(4))

print("\n[데이터 미리보기]")
display(train.head())


훈련 데이터: (9864, 19)
테스트 데이터: (2466, 18)

[자료형 및 결측 확인]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       9864 non-null   int64  
 1   Administrative           9864 non-null   int64  
 2   Administrative_Duration  9864 non-null   float64
 3   Informational            9864 non-null   int64  
 4   Informational_Duration   9864 non-null   float64
 5   ProductRelated           9864 non-null   int64  
 6   ProductRelated_Duration  9864 non-null   float64
 7   BounceRates              9864 non-null   float64
 8   ExitRates                9864 non-null   float64
 9   PageValues               9864 non-null   float64
 10  SpecialDay               9864 non-null   float64
 11  Month                    9864 non-null   object 
 12  OperatingSystems         9864 non-null   int64  
 13  Browser                 

,id,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,6,523.214286,2,263.0,48,2309.732468,0.003846,0.014904,10.600892,0.0,Mar,2,2,3,1,Returning_Visitor,False,1
1,2,2,20.200000,0,0.0,14,1261.316667,0.000000,0.009184,0.000000,0.0,Aug,2,2,1,5,Returning_Visitor,False,0
2,3,0,0.000000,0,0.0,3,71.500000,0.066667,0.133333,0.000000,0.0,Nov,2,2,3,3,Returning_Visitor,False,0
3,4,0,0.000000,0,0.0,5,50.500000,0.000000,0.040000,0.000000,0.0,Mar,2,2,1,1,Returning_Visitor,True,0
4,5,4,132.000000,1,23.0,24,1614.226190,0.000000,0.014286,0.000000,0.0,May,2,2,1,4,Returning_Visitor,False,0


In [3]:

from sklearn.model_selection import train_test_split

# 입력 변수(X)와 정답(y) 분리
X = train.drop(columns=['id', 'Revenue'])
y = train['Revenue']

# 학습용 80%, 독립 검증용 20% 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습 데이터:", X_train.shape)
print("검증 데이터:", X_valid.shape)

print("\n[학습 데이터 구매 비율]")
print(y_train.value_counts(normalize=True).round(4))

print("\n[검증 데이터 구매 비율]")
print(y_valid.value_counts(normalize=True).round(4))


학습 데이터: (7891, 17)
검증 데이터: (1973, 17)

[학습 데이터 구매 비율]
Revenue
0    0.8453
1    0.1547
Name: proportion, dtype: float64

[검증 데이터 구매 비율]
Revenue
0    0.8454
1    0.1546
Name: proportion, dtype: float64


In [4]:

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 문자형 변수
cat_cols = ['Month', 'VisitorType']

# 문자형 변수만 One-Hot Encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

# 학습 데이터로 인코딩 규칙 학습
X_train_encoded = preprocessor.fit_transform(X_train)

# 검증 데이터는 학습한 규칙만 적용
X_valid_encoded = preprocessor.transform(X_valid)

print("인코딩 전:", X_train.shape)
print("인코딩 후:", X_train_encoded.shape)

print("검증 데이터:", X_valid_encoded.shape)
print("변환된 변수 개수:", len(preprocessor.get_feature_names_out()))


인코딩 전: (7891, 17)
인코딩 후: (7891, 28)
검증 데이터: (1973, 28)
변환된 변수 개수: 28


In [5]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# 첫 번째 모델 생성
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# 학습 데이터로 모델 학습
rf_model.fit(X_train_encoded, y_train)

# 독립 검증 데이터 예측
rf_pred = rf_model.predict(X_valid_encoded)

# 성능 평가
rf_f1 = f1_score(y_valid, rf_pred)

print(f"RandomForest 검증 F1-score: {rf_f1:.4f}")
print("\n[분류 성능 상세]")
print(classification_report(y_valid, rf_pred))


RandomForest 검증 F1-score: 0.6761

[분류 성능 상세]
              precision    recall  f1-score   support

           0       0.96      0.90      0.93      1668
           1       0.60      0.78      0.68       305

    accuracy                           0.88      1973
   macro avg       0.78      0.84      0.80      1973
weighted avg       0.90      0.88      0.89      1973



In [6]:

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# 전처리 + 기존 RandomForest 설정 연결
rf_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', clone(rf_model))
])

# 구매 비율을 유지하는 5-Fold 교차검증
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 학습 데이터에서만 교차검증
rf_cv_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

print("[RandomForest 5-Fold F1-score]")
print(rf_cv_scores.round(4))

print(f"\nCV 평균 F1-score: {rf_cv_scores.mean():.4f}")
print(f"CV 표준편차: {rf_cv_scores.std():.4f}")
print(f"독립 검증 F1-score: {rf_f1:.4f}")


[RandomForest 5-Fold F1-score]
[0.7042 0.6557 0.6813 0.6849 0.649 ]

CV 평균 F1-score: 0.6750
CV 표준편차: 0.0202
독립 검증 F1-score: 0.6761


In [7]:

from sklearn.pipeline import Pipeline
from sklearn.base import clone

# 최종 제출용 모델
final_model_1 = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', clone(rf_model))
])

# 전체 훈련 데이터 준비
X_full = train.drop(columns=['id', 'Revenue'])
y_full = train['Revenue']

# 전체 데이터로 최종 학습
final_model_1.fit(X_full, y_full)

print("최종 모델 학습 완료!")
print("학습 데이터:", X_full.shape)
print("학습 정답:", y_full.shape)
print("모델:", type(final_model_1.named_steps['model']).__name__)
print("Pipeline 단계:", list(final_model_1.named_steps.keys()))


최종 모델 학습 완료!
학습 데이터: (9864, 17)
학습 정답: (9864,)
모델: RandomForestClassifier
Pipeline 단계: ['preprocessor', 'model']


In [8]:

import pickle
import os
import numpy as np

# 1. 최종 모델 저장
with open('answer_1.pkl', 'wb') as f:
    pickle.dump(final_model_1, f)

# 2. 저장된 모델 다시 불러오기
with open('answer_1.pkl', 'rb') as f:
    loaded_model_1 = pickle.load(f)

# 3. 저장 전후 예측 결과 비교
pred_before = final_model_1.predict(X_valid)
pred_after = loaded_model_1.predict(X_valid)

print("[PKL 파일 검증]")
print("파일 존재:", os.path.exists('answer_1.pkl'))
print("파일 크기:", os.path.getsize('answer_1.pkl'), "bytes")
print("복원 모델:", type(loaded_model_1).__name__)
print("예측 일치:", np.array_equal(pred_before, pred_after))
print("예측 행 개수:", len(pred_after))


[PKL 파일 검증]
파일 존재: True
파일 크기: 16225671 bytes
복원 모델: Pipeline
예측 일치: True
예측 행 개수: 1973


In [9]:

# 테스트 데이터 복사 (원본 보존)
answer_1 = test.copy()

# 모델 입력에서 id만 제외
X_test = test.drop(columns=['id'])

# 복원한 최종 모델로 예측
answer_1['Revenue'] = loaded_model_1.predict(X_test).astype(int)

# 제출 CSV 저장
answer_1.to_csv('answer_1.csv', index=False)

# 실제 저장된 파일 다시 읽기
check_1 = pd.read_csv('answer_1.csv')

print("[answer_1.csv 검증]")
print("파일 존재:", os.path.exists('answer_1.csv'))
print("데이터 크기:", check_1.shape)
print("마지막 컬럼:", check_1.columns[-1])
print("행 순서 및 원본 컬럼 일치:",
      check_1[test.columns].equals(test))
print("예측값 종류:", sorted(check_1['Revenue'].unique().tolist()))
print("예측값 결측:", check_1['Revenue'].isna().sum())

print("\n[예측값 분포]")
print(check_1['Revenue'].value_counts())

print("\n[제출 파일 미리보기]")
display(check_1.head())


[answer_1.csv 검증]
파일 존재: True
데이터 크기: (2466, 19)
마지막 컬럼: Revenue
행 순서 및 원본 컬럼 일치: True
예측값 종류: [0, 1]
예측값 결측: 0

[예측값 분포]
Revenue
0    1971
1     495
Name: count, dtype: int64

[제출 파일 미리보기]


,id,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,22,2189.500000,0.009524,0.028571,0.000000,0.0,Nov,4,2,1,10,Returning_Visitor,False,0
1,2,0,0.0,0,0.0,49,2110.124603,0.000000,0.004082,62.935492,0.0,Nov,2,2,1,10,Returning_Visitor,False,1
2,3,0,0.0,0,0.0,19,623.200000,0.022222,0.044444,0.000000,0.0,Jul,2,5,4,1,Returning_Visitor,True,0
3,4,0,0.0,0,0.0,7,236.000000,0.057143,0.085714,0.000000,0.0,Mar,2,2,3,2,Returning_Visitor,False,0
4,5,0,0.0,0,0.0,18,628.833333,0.000000,0.005556,0.000000,0.0,Mar,1,1,2,2,Returning_Visitor,True,0


<details><summary>힌트 1단계 (접근 방향) - 1번 문항</summary>

자료형·결측·양성(1) 비율부터 확인합니다. 번호·날짜 원문·정답과 같은 정보(누수) 열은 빼고, 범주형은 원-핫 인코딩합니다. '참고'의 파생 변수를 만든 뒤 후보 모델 두세 개를 **교차검증 F1** 로 비교하고, 양성이 적으면 **분류 임계값**도 교차검증으로 고르세요.

- 이 문항의 핵심: Month·VisitorType 은 문자열, Weekend 는 True/False 입니다. 원-핫 인코딩 또는 정수로 바꾸세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 1번 문항</summary>

사용할 도구: `pandas`, `joblib`, `LGBMClassifier`, `StratifiedKFold`, `cross_val_predict`, `f1_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 1번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from lightgbm import ____
from sklearn.model_selection import ____, ____
from sklearn.metrics import ____

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
LABEL = '____'
print(train[LABEL].value_counts(normalize=True).round(____).to_dict())

def feats(df):
    d = df.drop(columns=['____', LABEL], errors='____').copy()
    d['____'] = d['____'].astype(int)
    return pd.get_dummies(d, columns=['____', '____'], dtype=int)

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 분류 임계값을 교차검증으로 고르기
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_1.pkl`(모델) · `answer_1.csv`(예측)

</details>

## 2. Text 문제 - 문장 속 두 개체의 관계 분류(6종) (35점)

<sub>학습 목표: 문장 속 두 개체의 관계 분류(6종) 모델로 테스트 정확도 0.9 이상을 달성할 수 있다 | 역량: PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | 기술: pandas, scikit-learn(ColumnTransformer, Pipeline, OneHotEncoder, TfidfVectorizer, LogisticRegression, LinearSVC), joblib | 난이도: 중급 | 예상 시간: 60분</sub>

### 문제 개요

한 문장 안의 **주어 개체(subject)** 와 **목적 개체(object)** 사이의 관계를 6종(소속·대표·생년월일·출신·직함 등) 중에서 고르는 관계 추출(Relation Extraction) 문제입니다. 문장 텍스트뿐 아니라 **개체의 종류(인물 PER · 단체 ORG · 날짜 DAT · 지역 LOC · 직함 POH 등)** 가 큰 단서입니다.

제공된 훈련 데이터셋을 활용해 텍스트 다중 분류(관계 6종) 모델을 개발하고, 테스트 데이터셋의 'relation' 값을 예측하시기 바랍니다.

### 달성 목표

"02_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 90% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `02_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `02_test_x.csv`: 테스트 데이터셋입니다. 라벨('relation') 값이 제외되어 있습니다.

**Label 정보**

- 'relation' : org:member_of: 단체의 소속(상위 단체) / org:top_members/employees: 단체의 대표·구성원 / per:date_of_birth: 인물의 생년월일 / per:employee_of: 인물의 소속 / per:origin: 인물의 출신(국가·지역) / per:title: 인물의 직함·직업

**Data 정보**

- 'sentence' : 문장
- 'subject · subject_type' : 주어 개체의 글자 · 종류
- 'object · object_type' : 목적 개체의 글자 · 종류
- 'id' : 테스트 행 번호(테스트에만)

**데이터 출처** — KLUE-RE 관계 추출(HF klue/klue · CC BY-SA 4.0 — 같은 조건으로만 재배포) 중 6종 — 위키백과·뉴스 문장, 관계마다 1,000건

### 데이터 예시

`02_train.csv`

| sentence | subject | subject_type | object | object_type | relation |
|---|---|---|---|---|---|
| 미국의 거대 패스트푸드 업체는 러시아 진출 30주년을 맞아 3루블에 맥도날드의 상징적인 버거를 제공하는 기념 할인행사를 할 계획이었다. | 맥도날드 | ORG | 패스트푸드 | POH | org:member_of |
| 남순은 "감스트, 외질 관련해서는 나락즈 복귀하면 시원하게 말씀드리겠다. 그 친구들이 지금은 복귀를 안 했기 때문에 제가 이러쿵저러쿵 얘기하면 그 친구들에게 또 다른 피해를 줄까 봐 언급할 수가 없다"라며 자숙 원인이었던 지난 6월 나락즈 방송에 대해서는 말을 아꼈다. | 나락즈 | PER | 감스트 | PER | per:employee_of |
| 도쿠가와 이에노부(1662년 6월 11일 - 1712년 11월 12일)는 일본 에도 막부 제6대 쇼군(재위 1709년 - 1712년)이다. | 도쿠가와 이에노부 | PER | 1662년 6월 11일 | DAT | per:date_of_birth |

`02_test_x.csv`

| id | sentence | subject | subject_type | object | object_type |
|---|---|---|---|---|---|
| 1 | 강기갑은 1953년 6월 7일 경상남도 사천군 사천면 장전2리에서 강진규(姜振圭, - 2005년 7월 17일)의 셋째 아들로 태어났다. | 강기갑 | PER | 1953년 6월 7일 | DAT |
| 2 | 휘그당은 다시 윌리엄 헨리 해리슨을 대통령 후보로 지명하였고, 버지니아 주의 전 상원 존 타일러를 그의 러닝메이트로 선택하였다. | 존 타일러 | PER | 휘그당 | ORG |
| 3 | 천안 현대캐피탈 스카이워커스는 수원 KEPCO 45로부터 문성민을 영입하는 대신 센터 하경민과 레프트 임시형을 내주게 된다. | 문성민 | PER | 천안 현대캐피탈 스카이워커스 | ORG |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 문장 속 두 개체의 관계 분류(6종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_2.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('relation') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_2.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'relation' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'relation' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 문장은 TF-IDF, 개체 종류(subject_type·object_type)는 원-핫으로 바꿔 `ColumnTransformer` 로 이어 붙이세요.
- 개체 글자(object)에도 단서가 있습니다(예: '1980년' → 생년월일, '감독' → 직함). object 열에 문자 n-gram 을 따로 적용해 보세요.
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트 행 순서(`id`)를 유지하고 관계 이름(예: per:title)을 그대로 제출하세요.

### 채점 진행 방식 안내

- 2번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_2.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 81% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [11]:

import pandas as pd

# 2번 데이터 불러오기
train_2 = pd.read_csv('02_train.csv')
test_2 = pd.read_csv('02_test_x.csv')

print("[데이터 크기]")
print("훈련:", train_2.shape)
print("테스트:", test_2.shape)

print("\n[컬럼 및 자료형]")
train_2.info()

print("\n[정답 분포]")
print(train_2['relation'].value_counts())

print("\n[개체 유형 조합별 관계]")
print(pd.crosstab(
    [train_2['subject_type'], train_2['object_type']],
    train_2['relation']
))

print("\n[데이터 미리보기]")
display(train_2.head(5))


[데이터 크기]
훈련: (4800, 6)
테스트: (1200, 6)

[컬럼 및 자료형]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4800 entries, 0 to 4799
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   sentence      4800 non-null   object
 1   subject       4800 non-null   object
 2   subject_type  4800 non-null   object
 3   object        4800 non-null   object
 4   object_type   4800 non-null   object
 5   relation      4800 non-null   object
dtypes: object(6)
memory usage: 225.1+ KB

[정답 분포]
relation
org:member_of                800
per:employee_of              800
per:date_of_birth            800
per:origin                   800
per:title                    800
org:top_members/employees    800
Name: count, dtype: int64

[개체 유형 조합별 관계]
relation                  org:member_of  org:top_members/employees  \
subject_type object_type                                             
ORG          DAT                      3                          1  

,sentence,subject,subject_type,object,object_type,relation
0,미국의 거대 패스트푸드 업체는 러시아 진출 30주년을 맞아 3루블에 맥도날드의 상징...,맥도날드,ORG,패스트푸드,POH,org:member_of
1,"남순은 ""감스트, 외질 관련해서는 나락즈 복귀하면 시원하게 말씀드리겠다. 그 친구들...",나락즈,PER,감스트,PER,per:employee_of
2,도쿠가와 이에노부(1662년 6월 11일 - 1712년 11월 12일)는 일본 에도...,도쿠가와 이에노부,PER,1662년 6월 11일,DAT,per:date_of_birth
3,"예를 들면 우고 차베스 베네수엘라 대통령은 남남 협력을 ""21세기 사회주의""라는 그...",우고 차베스,PER,베네수엘라,LOC,per:origin
4,'코로나19' 감염 극복을 위한 '사회적 거리두기' 캠페인이 이뤄지는데도 KBS 9...,최동석,PER,아나운서,POH,per:title


In [12]:

from sklearn.model_selection import train_test_split

# 입력 변수와 정답 분리
X_2 = train_2.drop(columns=['relation'])
y_2 = train_2['relation']

# 학습용 80%, 독립 검증용 20%
X_train_2, X_valid_2, y_train_2, y_valid_2 = train_test_split(
    X_2,
    y_2,
    test_size=0.2,
    random_state=42,
    stratify=y_2
)

print("[데이터 분리 결과]")
print("학습:", X_train_2.shape)
print("검증:", X_valid_2.shape)

print("\n[학습 데이터 관계별 개수]")
print(y_train_2.value_counts())

print("\n[검증 데이터 관계별 개수]")
print(y_valid_2.value_counts())


[데이터 분리 결과]
학습: (3840, 5)
검증: (960, 5)

[학습 데이터 관계별 개수]
relation
per:title                    640
per:date_of_birth            640
org:top_members/employees    640
per:origin                   640
per:employee_of              640
org:member_of                640
Name: count, dtype: int64

[검증 데이터 관계별 개수]
relation
org:member_of                160
per:date_of_birth            160
per:title                    160
org:top_members/employees    160
per:origin                   160
per:employee_of              160
Name: count, dtype: int64


In [13]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 텍스트 + 개체 유형 전처리
preprocessor_2 = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(
            analyzer='char',
            ngram_range=(2, 4),
            min_df=2,
            max_features=30000,
            sublinear_tf=True
        ), 'sentence'),

        ('entity_type', OneHotEncoder(
            handle_unknown='ignore'
        ), ['subject_type', 'object_type'])
    ]
)

# 학습 데이터에서만 전처리 규칙 학습
X_train_2_encoded = preprocessor_2.fit_transform(X_train_2)

# 검증 데이터에는 동일한 규칙 적용
X_valid_2_encoded = preprocessor_2.transform(X_valid_2)

print("[전처리 결과]")
print("학습:", X_train_2_encoded.shape)
print("검증:", X_valid_2_encoded.shape)
print("데이터 타입:", type(X_train_2_encoded).__name__)

print("\n[특징 개수]")
print("전체 특징:", X_train_2_encoded.shape[1])


[전처리 결과]
학습: (3840, 30008)
검증: (960, 30008)
데이터 타입: csr_matrix

[특징 개수]
전체 특징: 30008


In [14]:

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# 첫 번째 텍스트 분류 모델
svc_model_2 = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=42
)

# 모델 학습
svc_model_2.fit(X_train_2_encoded, y_train_2)

# 독립 검증 데이터 예측
svc_pred_2 = svc_model_2.predict(X_valid_2_encoded)

# 성능 평가
svc_acc_2 = accuracy_score(y_valid_2, svc_pred_2)

print(f"LinearSVC 검증 Accuracy: {svc_acc_2:.4f}")
print("\n[관계별 분류 성능]")
print(classification_report(y_valid_2, svc_pred_2))


LinearSVC 검증 Accuracy: 0.9292

[관계별 분류 성능]
                           precision    recall  f1-score   support

            org:member_of       0.97      0.99      0.98       160
org:top_members/employees       0.99      0.96      0.98       160
        per:date_of_birth       0.98      1.00      0.99       160
          per:employee_of       0.83      0.90      0.86       160
               per:origin       0.88      0.80      0.84       160
                per:title       0.93      0.92      0.92       160

                 accuracy                           0.93       960
                macro avg       0.93      0.93      0.93       960
             weighted avg       0.93      0.93      0.93       960



In [15]:

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# 전처리 + LinearSVC 연결
svc_pipeline_2 = Pipeline([
    ('preprocessor', clone(preprocessor_2)),
    ('model', clone(svc_model_2))
])

# 5-Fold 교차검증
cv_2 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 학습 데이터에서만 교차검증
svc_cv_scores_2 = cross_val_score(
    svc_pipeline_2,
    X_train_2,
    y_train_2,
    cv=cv_2,
    scoring='accuracy',
    n_jobs=-1
)

print("[LinearSVC 5-Fold Accuracy]")
print(svc_cv_scores_2.round(4))

print(f"\nCV 평균 Accuracy: {svc_cv_scores_2.mean():.4f}")
print(f"CV 표준편차: {svc_cv_scores_2.std():.4f}")
print(f"독립 검증 Accuracy: {svc_acc_2:.4f}")


[LinearSVC 5-Fold Accuracy]
[0.9323 0.9049 0.9154 0.9206 0.9323]

CV 평균 Accuracy: 0.9211
CV 표준편차: 0.0104
독립 검증 Accuracy: 0.9292


In [16]:

from sklearn.pipeline import Pipeline
from sklearn.base import clone

# 최종 제출용 모델
final_model_2 = Pipeline([
    ('preprocessor', clone(preprocessor_2)),
    ('model', clone(svc_model_2))
])

# 전체 훈련 데이터 준비
X_full_2 = train_2.drop(columns=['relation'])
y_full_2 = train_2['relation']

# 전체 데이터로 최종 학습
final_model_2.fit(X_full_2, y_full_2)

print("[최종 모델 학습 완료]")
print("학습 데이터:", X_full_2.shape)
print("학습 정답:", y_full_2.shape)
print("모델:", type(final_model_2.named_steps['model']).__name__)
print("Pipeline 단계:", list(final_model_2.named_steps.keys()))
print("분류 클래스:", final_model_2.classes_)


[최종 모델 학습 완료]
학습 데이터: (4800, 5)
학습 정답: (4800,)
모델: LinearSVC
Pipeline 단계: ['preprocessor', 'model']
분류 클래스: ['org:member_of' 'org:top_members/employees' 'per:date_of_birth'
 'per:employee_of' 'per:origin' 'per:title']


In [17]:

import pickle
import os
import numpy as np

# 1. 최종 모델 저장
with open('answer_2.pkl', 'wb') as f:
    pickle.dump(final_model_2, f)

# 2. 저장된 모델 다시 불러오기
with open('answer_2.pkl', 'rb') as f:
    loaded_model_2 = pickle.load(f)

# 3. 저장 전후 예측 결과 비교
pred_before_2 = final_model_2.predict(X_valid_2)
pred_after_2 = loaded_model_2.predict(X_valid_2)

print("[PKL 파일 검증]")
print("파일 존재:", os.path.exists('answer_2.pkl'))
print("파일 크기:", os.path.getsize('answer_2.pkl'), "bytes")
print("복원 모델:", type(loaded_model_2).__name__)
print("예측 일치:", np.array_equal(pred_before_2, pred_after_2))
print("예측 행 개수:", len(pred_after_2))
print("분류 클래스:", loaded_model_2.classes_)


[PKL 파일 검증]
파일 존재: True
파일 크기: 2559260 bytes
복원 모델: Pipeline
예측 일치: True
예측 행 개수: 960
분류 클래스: ['org:member_of' 'org:top_members/employees' 'per:date_of_birth'
 'per:employee_of' 'per:origin' 'per:title']


In [18]:

# 원본 테스트 데이터 보존
answer_2 = test_2.copy()

# 모델 입력에서 id 제외
X_test_2 = test_2.drop(columns=['id'])

# 저장된 모델로 관계 예측
answer_2['relation'] = loaded_model_2.predict(X_test_2)

# 제출 파일 저장
answer_2.to_csv('answer_2.csv', index=False)

# 실제 저장된 CSV 다시 읽어 검증
check_2 = pd.read_csv('answer_2.csv')

print("[answer_2.csv 검증]")
print("파일 존재:", os.path.exists('answer_2.csv'))
print("데이터 크기:", check_2.shape)
print("마지막 컬럼:", check_2.columns[-1])
print("원본 컬럼 및 행 순서 일치:",
      check_2[test_2.columns].equals(test_2))
print("예측값 종류:", sorted(check_2['relation'].unique().tolist()))
print("잘못된 라벨 개수:",
      (~check_2['relation'].isin(train_2['relation'].unique())).sum())
print("예측값 결측:", check_2['relation'].isna().sum())

print("\n[예측 관계별 개수]")
print(check_2['relation'].value_counts())

print("\n[제출 파일 미리보기]")
display(check_2.head())


[answer_2.csv 검증]
파일 존재: True
데이터 크기: (1200, 7)
마지막 컬럼: relation
원본 컬럼 및 행 순서 일치: True
예측값 종류: ['org:member_of', 'org:top_members/employees', 'per:date_of_birth', 'per:employee_of', 'per:origin', 'per:title']
잘못된 라벨 개수: 0
예측값 결측: 0

[예측 관계별 개수]
relation
per:employee_of              217
per:date_of_birth            207
org:member_of                207
per:title                    203
org:top_members/employees    194
per:origin                   172
Name: count, dtype: int64

[제출 파일 미리보기]


,id,sentence,subject,subject_type,object,object_type,relation
0,1,"강기갑은 1953년 6월 7일 경상남도 사천군 사천면 장전2리에서 강진규(姜振圭, ...",강기갑,PER,1953년 6월 7일,DAT,per:date_of_birth
1,2,"휘그당은 다시 윌리엄 헨리 해리슨을 대통령 후보로 지명하였고, 버지니아 주의 전 상...",존 타일러,PER,휘그당,ORG,per:employee_of
2,3,천안 현대캐피탈 스카이워커스는 수원 KEPCO 45로부터 문성민을 영입하는 대신 센...,문성민,PER,천안 현대캐피탈 스카이워커스,ORG,per:employee_of
3,4,만지케르트 전투는 1071년 8월 26일에 비잔티움 제국과 알프 아르슬란이 지휘하는...,알프 아르슬란,PER,셀주크 제국,ORG,per:employee_of
4,5,리사 디 배나는 중국에서 개최된 2007년 FIFA 여자 월드컵에서 노르웨이와의 조...,리사 디 배나,PER,오스트레일리아,ORG,per:origin


<details><summary>힌트 1단계 (접근 방향) - 2번 문항</summary>

라벨 분포와 문장 길이를 확인합니다. 문장을 TF-IDF 로 바꾸되 **단어 단위와 글자 조각(char n-gram)** 을 비교하고, 나이브 베이즈·선형 SVM·로지스틱 회귀 중 **교차검증 정확도** 가 높은 조합으로 전체 학습하세요. 다른 컬럼(숫자·범주)이 단서이면 `ColumnTransformer` 로 함께 넣습니다.

- 이 문항의 핵심: 문장은 TF-IDF, 개체 종류(subject_type·object_type)는 원-핫으로 바꿔 `ColumnTransformer` 로 이어 붙이세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 2번 문항</summary>

사용할 도구: `pandas`, `joblib`, `ColumnTransformer`, `Pipeline`, `OneHotEncoder`, `TfidfVectorizer`, `LogisticRegression`, `LinearSVC`, `StratifiedKFold`, `cross_val_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 2번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from sklearn.compose import ____
from sklearn.pipeline import ____
from sklearn.preprocessing import ____
from sklearn.feature_extraction.text import ____
from sklearn.linear_model import ____
from sklearn.svm import ____
from sklearn.model_selection import ____, ____

train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')
LABEL = '____'
COLS = ['____', '____', '____', '____', '____']

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_2.pkl`(모델) · `answer_2.csv`(예측)

</details>

## 3. Image 문제 - 고양이 사진으로 품종 분류(5종) (35점)

<sub>학습 목표: 고양이 사진으로 품종 분류(5종) 모델로 테스트 정확도 0.8 이상을 달성할 수 있다 | 역량: PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | 기술: pandas, scikit-learn(LogisticRegression), Keras(TensorFlow.js), joblib | 난이도: 실전 | 예상 시간: 60분</sub>

### 문제 개요

고양이 사진으로 **품종 5종**(아비시니안·벵갈·봄베이·브리티시 쇼트헤어·러시안 블루)을 분류하는 문제입니다. 회색 털(브리티시·러시안 블루)과 검은 털(봄베이)처럼 비슷한 품종이 섞여 있고 품종당 사진이 약 100장뿐이라, ImageNet 으로 학습된 모델의 특징을 쓰는 **전이학습** 이 효과적입니다.

제공된 훈련 데이터셋을 활용해 이미지 다중 분류(5종) 모델을 개발하고, 테스트 데이터셋의 'label' 값을 예측하시기 바랍니다.

### 달성 목표

"03_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 80% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `03_images.zip`: 압축을 풀면 `03_train/<클래스>/cat_*.png`(학습)와 `03_test/test_*.png`(테스트) 폴더가 생깁니다. 128×128 컬러 PNG.
- `03_train.csv`: 학습 이미지 파일명과 라벨
- `03_test_x.csv`: 테스트 이미지 파일명(라벨 제외)

**Label 정보**

- 'label' : abyssinian: 아비시니안 / bengal: 벵갈 / bombay: 봄베이 / british_shorthair: 브리티시 쇼트헤어 / russian_blue: 러시안 블루

**데이터 출처** — Oxford-IIIT Pet(HF timm/oxford-iiit-pet · CC BY-SA 4.0 — 같은 조건으로만 재배포) 중 고양이 5품종 — 가운데 정사각형 128×128

### 데이터 예시

학습 이미지 라벨별 장수: abyssinian 80, bengal 80, british_shorthair 80, russian_blue 80, bombay 76

| file_name | label |
|---|---|
| cat_00001.png | bombay |
| cat_00002.png | abyssinian |
| cat_00003.png | bengal |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 고양이 사진으로 품종 분류(5종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_3.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('label') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_3.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'label' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'label' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 주피터라이트에는 **MobileNet v1(alpha=0.25, 224×224, ImageNet) 가중치가 미리 들어 있습니다**. `MobileNet(input_shape=(224, 224, 3), alpha=0.25, include_top=False, weights='imagenet', pooling='avg')` 로 특징만 뽑고(가중치 고정), 그 특징으로 로지스틱 회귀 같은 분류기를 학습하세요. 다른 사전학습 모델(MobileNetV2·ResNet 등)은 브라우저에서 쓸 수 없습니다.
- MobileNet 전용 `preprocess_input` 을 쓰고, 1/255 rescale 을 **함께 하지 마세요**(이중 정규화).
- 브라우저에서는 224×224 이미지를 한 번에 많이 올리면 느려집니다. **64장씩 나눠** 특징을 뽑으세요.
- 품종당 학습 이미지가 80장 정도로 적습니다. 직접 만든 CNN 보다 MobileNet 특징 + 로지스틱 회귀가 훨씬 정확합니다. 분류기 규제(C)를 바꿔 비교해 보세요.
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트는 `03_test_x.csv` 의 파일명 순서대로 읽어 예측하세요.

### 채점 진행 방식 안내

- 3번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_3.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 72% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [20]:

import os
import zipfile
import pandas as pd

# 3번 데이터 불러오기
train_3 = pd.read_csv('03_train.csv')
test_3 = pd.read_csv('03_test_x.csv')

# 이미지 압축 해제 경로
IMG_DIR = os.environ.get('AICE_IMG_DIR', os.getcwd())

train_dir_3 = os.path.join(IMG_DIR, '03_train')
test_dir_3 = os.path.join(IMG_DIR, '03_test')

# 아직 압축을 풀지 않은 경우에만 실행
if not (os.path.isdir(train_dir_3) and os.path.isdir(test_dir_3)):
    with zipfile.ZipFile('03_images.zip') as z:
        z.extractall(IMG_DIR)

print("[데이터 크기]")
print("훈련:", train_3.shape)
print("테스트:", test_3.shape)

print("\n[품종별 이미지 수]")
print(train_3['label'].value_counts())

print("\n[이미지 폴더 확인]")
print("훈련 폴더:", os.path.isdir(train_dir_3))
print("테스트 폴더:", os.path.isdir(test_dir_3))

print("\n[데이터 미리보기]")
display(train_3.head())


[데이터 크기]
훈련: (396, 2)
테스트: (100, 1)

[품종별 이미지 수]
label
abyssinian           80
british_shorthair    80
bengal               80
russian_blue         80
bombay               76
Name: count, dtype: int64

[이미지 폴더 확인]
훈련 폴더: True
테스트 폴더: True

[데이터 미리보기]


,file_name,label
0,cat_00001.png,bombay
1,cat_00002.png,abyssinian
2,cat_00003.png,bengal
3,cat_00004.png,abyssinian
4,cat_00005.png,bombay


In [21]:

from PIL import Image
import os

# CSV에 기록된 순서대로 이미지 경로 만들기
train_paths_3 = [
    os.path.join(train_dir_3, row.label, row.file_name)
    for row in train_3.itertuples(index=False)
]

test_paths_3 = [
    os.path.join(test_dir_3, name)
    for name in test_3['file_name']
]

# 실제 파일 존재 여부
all_paths_3 = train_paths_3 + test_paths_3
missing_3 = [p for p in all_paths_3 if not os.path.isfile(p)]

# 이미지 크기, 색상 형식, 로딩 오류 확인
sizes_3 = set()
modes_3 = set()
errors_3 = []

for path in all_paths_3:
    if not os.path.isfile(path):
        continue
    try:
        with Image.open(path) as img:
            sizes_3.add(img.size)
            modes_3.add(img.mode)
            img.load()
    except Exception as e:
        errors_3.append((path, str(e)))

print("[이미지 경로 검증]")
print("훈련 이미지:", len(train_paths_3))
print("테스트 이미지:", len(test_paths_3))
print("누락 파일:", len(missing_3))
print("이미지 로딩 오류:", len(errors_3))

print("\n[이미지 속성]")
print("이미지 크기:", sizes_3)
print("이미지 색상 형식:", modes_3)

print("\n[이미지 경로 예시]")
print("훈련:", train_paths_3[0])
print("테스트:", test_paths_3[0])

if missing_3:
    print("누락 예시:", missing_3[:3])
if errors_3:
    print("오류 예시:", errors_3[:3])


[이미지 경로 검증]
훈련 이미지: 396
테스트 이미지: 100
누락 파일: 0
이미지 로딩 오류: 0

[이미지 속성]
이미지 크기: {(128, 128)}
이미지 색상 형식: {'RGB'}

[이미지 경로 예시]
훈련: /content/03_train/bombay/cat_00001.png
테스트: /content/03_test/test_0001.png


In [22]:

import numpy as np
from PIL import Image
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input

# ImageNet 사전학습 MobileNet 불러오기
mobilenet_3 = MobileNet(
    input_shape=(224, 224, 3),
    alpha=0.25,
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

# 사전학습 가중치 고정
mobilenet_3.trainable = False

# 첫 번째 훈련 이미지로 정상 작동 확인
with Image.open(train_paths_3[0]) as img:
    img = img.convert('RGB').resize((224, 224))
    sample_3 = np.asarray(img, dtype=np.float32)

sample_3 = np.expand_dims(sample_3, axis=0)
sample_3 = preprocess_input(sample_3)

# 특징 추출 테스트
sample_features_3 = mobilenet_3.predict(
    sample_3,
    verbose=0
)

print("[MobileNet 확인]")
print("모델:", mobilenet_3.name)
print("입력 형태:", sample_3.shape)
print("특징 형태:", sample_features_3.shape)
print("학습 가능 여부:", mobilenet_3.trainable)
print("NaN 개수:", np.isnan(sample_features_3).sum())


2108140/2108140 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[MobileNet 확인]
모델: mobilenet_0.25_224
입력 형태: (1, 224, 224, 3)
특징 형태: (1, 256)
학습 가능 여부: False
NaN 개수: 0


In [23]:

import numpy as np
from PIL import Image
from tensorflow.keras.applications.mobilenet import preprocess_input

# 이미지 경로 목록을 입력받아 특징을 추출하는 함수
def extract_features_3(paths, batch_size=64):
    features = []

    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        images = []

        for path in batch_paths:
            with Image.open(path) as img:
                img = img.convert('RGB').resize((224, 224))
                images.append(np.asarray(img, dtype=np.float32))

        # 이미지 묶음 생성 및 MobileNet 전처리
        batch = np.stack(images)
        batch = preprocess_input(batch)

        # MobileNet으로 특징 추출
        batch_features = mobilenet_3.predict(
            batch,
            verbose=0,
            batch_size=32
        )

        features.append(batch_features)

    return np.concatenate(features, axis=0)

# 훈련 이미지 특징 추출
X_img_train_3 = extract_features_3(train_paths_3)

# 테스트 이미지 특징 추출
X_img_test_3 = extract_features_3(test_paths_3)

# 훈련 라벨
y_img_3 = train_3['label'].to_numpy()

print("[전체 이미지 특징 추출 완료]")
print("훈련 특징:", X_img_train_3.shape)
print("테스트 특징:", X_img_test_3.shape)
print("훈련 라벨:", y_img_3.shape)

print("\n[데이터 검증]")
print("훈련 NaN:", np.isnan(X_img_train_3).sum())
print("테스트 NaN:", np.isnan(X_img_test_3).sum())
print("분류 품종:", np.unique(y_img_3))


[전체 이미지 특징 추출 완료]
훈련 특징: (396, 256)
테스트 특징: (100, 256)
훈련 라벨: (396,)

[데이터 검증]
훈련 NaN: 0
테스트 NaN: 0
분류 품종: ['abyssinian' 'bengal' 'bombay' 'british_shorthair' 'russian_blue']


In [24]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 학습 80%, 독립 검증 20%
X_train_3, X_valid_3, y_train_3, y_valid_3 = train_test_split(
    X_img_train_3,
    y_img_3,
    test_size=0.2,
    random_state=42,
    stratify=y_img_3
)

print("[데이터 분리]")
print("학습:", X_train_3.shape)
print("검증:", X_valid_3.shape)

# 첫 번째 분류 모델
lr_model_3 = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

# 모델 학습
lr_model_3.fit(X_train_3, y_train_3)

# 독립 검증
lr_pred_3 = lr_model_3.predict(X_valid_3)
lr_acc_3 = accuracy_score(y_valid_3, lr_pred_3)

print(f"\nLogisticRegression 검증 Accuracy: {lr_acc_3:.4f}")

print("\n[품종별 분류 성능]")
print(classification_report(y_valid_3, lr_pred_3))


[데이터 분리]
학습: (316, 256)
검증: (80, 256)

LogisticRegression 검증 Accuracy: 0.8500

[품종별 분류 성능]
                   precision    recall  f1-score   support

       abyssinian       0.78      0.88      0.82        16
           bengal       0.87      0.81      0.84        16
           bombay       1.00      1.00      1.00        16
british_shorthair       0.92      0.75      0.83        16
     russian_blue       0.72      0.81      0.76        16

         accuracy                           0.85        80
        macro avg       0.86      0.85      0.85        80
     weighted avg       0.86      0.85      0.85        80



In [25]:

from sklearn.linear_model import LogisticRegression

# 최종 제출용 분류 모델
final_model_3 = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

# 전체 396장 이미지의 기존 특징으로 학습
final_model_3.fit(X_img_train_3, y_img_3)

print("[최종 모델 학습 완료]")
print("학습 데이터:", X_img_train_3.shape)
print("학습 정답:", y_img_3.shape)
print("최종 모델:", type(final_model_3).__name__)
print("분류 클래스:", final_model_3.classes_)
print("모델 입력 특징 수:", final_model_3.n_features_in_)


[최종 모델 학습 완료]
학습 데이터: (396, 256)
학습 정답: (396,)
최종 모델: LogisticRegression
분류 클래스: ['abyssinian' 'bengal' 'bombay' 'british_shorthair' 'russian_blue']
모델 입력 특징 수: 256


In [26]:

import pickle
import os
import numpy as np

# 1. 최종 LogisticRegression 모델 저장
with open('answer_3.pkl', 'wb') as f:
    pickle.dump(final_model_3, f)

# 2. 실제 저장된 모델 다시 불러오기
with open('answer_3.pkl', 'rb') as f:
    loaded_model_3 = pickle.load(f)

# 3. 저장 전후 테스트 이미지 예측 비교
pred_before_3 = final_model_3.predict(X_img_test_3)
pred_after_3 = loaded_model_3.predict(X_img_test_3)

print("[PKL 파일 검증]")
print("파일 존재:", os.path.exists('answer_3.pkl'))
print("파일 크기:", os.path.getsize('answer_3.pkl'), "bytes")
print("복원 모델:", type(loaded_model_3).__name__)
print("예측 일치:", np.array_equal(pred_before_3, pred_after_3))
print("예측 행 개수:", len(pred_after_3))
print("분류 클래스:", loaded_model_3.classes_)


[PKL 파일 검증]
파일 존재: True
파일 크기: 11047 bytes
복원 모델: LogisticRegression
예측 일치: True
예측 행 개수: 100
분류 클래스: ['abyssinian' 'bengal' 'bombay' 'british_shorthair' 'russian_blue']


In [27]:

# 1. 원본 테스트 데이터 복사
answer_3 = test_3.copy()

# 2. 복원한 최종 모델로 품종 예측
answer_3['label'] = loaded_model_3.predict(X_img_test_3)

# 3. 제출 CSV 저장
answer_3.to_csv('answer_3.csv', index=False)

# 4. 저장된 파일 다시 읽어 검증
check_3 = pd.read_csv('answer_3.csv')

print("[answer_3.csv 검증]")
print("파일 존재:", os.path.exists('answer_3.csv'))
print("데이터 크기:", check_3.shape)
print("컬럼:", check_3.columns.tolist())
print("원본 파일명 및 순서 일치:",
      check_3['file_name'].equals(test_3['file_name']))
print("예측값 종류:", sorted(check_3['label'].unique().tolist()))
print("잘못된 라벨 개수:",
      (~check_3['label'].isin(train_3['label'].unique())).sum())
print("예측값 결측:", check_3['label'].isna().sum())
print("모델 예측과 CSV 일치:",
      np.array_equal(check_3['label'].to_numpy(), pred_after_3))

print("\n[품종별 예측 개수]")
print(check_3['label'].value_counts())

print("\n[제출 파일 미리보기]")
display(check_3.head())


[answer_3.csv 검증]
파일 존재: True
데이터 크기: (100, 2)
컬럼: ['file_name', 'label']
원본 파일명 및 순서 일치: True
예측값 종류: ['abyssinian', 'bengal', 'bombay', 'british_shorthair', 'russian_blue']
잘못된 라벨 개수: 0
예측값 결측: 0
모델 예측과 CSV 일치: True

[품종별 예측 개수]
label
bombay               23
bengal               22
abyssinian           20
russian_blue         19
british_shorthair    16
Name: count, dtype: int64

[제출 파일 미리보기]


,file_name,label
0,test_0001.png,british_shorthair
1,test_0002.png,russian_blue
2,test_0003.png,bengal
3,test_0004.png,british_shorthair
4,test_0005.png,abyssinian


<details><summary>힌트 1단계 (접근 방향) - 3번 문항</summary>

압축을 풀고 폴더 구조(학습: 클래스별 폴더, 테스트: 한 폴더)를 확인합니다. 이미지를 같은 크기로 읽어 0~1 로 맞추고, 검증 데이터를 떼어 **작은 CNN**(또는 사전학습 모델 특징 + 분류기)을 학습합니다. 증강·Dropout·조기 종료로 과적합을 막고, 테스트는 `03_test_x.csv` 순서대로 예측하세요.

- 이 문항의 핵심: 주피터라이트에는 **MobileNet v1(alpha=0.25, 224×224, ImageNet) 가중치가 미리 들어 있습니다**. `MobileNet(input_shape=(224, 224, 3), alpha=0.25, include_top=False, weights='imagenet', pooling='avg')` 로 특징만 뽑고(가중치 고정), 그 특징으로 로지스틱 회귀 같은 분류기를 학습하세요. 다른 사전학습 모델(MobileNetV2·ResNet 등)은 브라우저에서 쓸 수 없습니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 3번 문항</summary>

사용할 도구: `pandas`, `joblib`, `Image`, `LogisticRegression`, `train_test_split`, `accuracy_score`, `MobileNet`, `preprocess_input`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html>
- 참고 문서: <https://keras.io/api/applications/mobilenet/>

</details>

<details><summary>힌트 3단계 (코드 구조) - 3번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
import ____
from PIL import ____
from sklearn.linear_model import ____
from sklearn.model_selection import ____
from sklearn.metrics import ____
from tensorflow.keras.applications import ____
from tensorflow.keras.applications.mobilenet import ____

IMG_DIR = os.environ.get('____', '____')        # 압축을 풀 위치(브라우저판은 메모리 폴더 /tmp)
if not os.path.exists(os.path.join(IMG_DIR, '____')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)
train = pd.read_csv('03_train.csv')
test = pd.read_csv('03_test_x.csv')

# ▷ 모델 구성(CNN 또는 사전학습 특징 추출)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_3.pkl`(모델) · `answer_3.csv`(예측)

</details>